In [7]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

def run_interactive_fake_door_nice_blue(visitors, conversions, target_cvr):
    # 1. Bayesian Update (The Math)
    prior_alpha = 1
    prior_beta = 1
    
    post_alpha = prior_alpha + conversions
    post_beta = prior_beta + (visitors - conversions)
    
    # 2. Calculate Statistics
    mean_rate = post_alpha / (post_alpha + post_beta)
    
    # Calculate 95% Credible Interval
    ci_lower = beta.ppf(0.025, post_alpha, post_beta) 
    ci_upper = beta.ppf(0.975, post_alpha, post_beta) 
    
    # Decision: P(CVR > target)
    prob_above_target = 1 - beta.cdf(target_cvr, post_alpha, post_beta)
    cdf_value = beta.cdf(target_cvr, post_alpha, post_beta)
    verdict = "続行" if prob_above_target >= 0.95 else "証拠不十分"

    # 3. Print Summary
    print(f"--- フェイクドアテスト ---")
    print(f"手法   : ベイズ推定 ベータ二択分布 モデル）")
    print(f"         Beta = CVRのような確率の分布を表す型。")
    print(f"         Binomial = 「成功 or 失敗」の試行を繰り返すデータの型。")
    print(f"         → 訪問者が「CVするかしないか」のデータ(二択)から、")
    print(f"           真のCVRの確率分布(Beta)を推定するモデル。")
    print(f"         事前分布を無情報（一様分布）とし、観測データから")
    print(f"         真のCVRに対する信念を更新する。")
    print(f"KPI    : コンバージョン率（CVR）")
    print(f"目標   : 事業成立ライン {target_cvr:.1%}")
    print(f"-" * 45)
    print(f"実績   : {visitors}人中 {conversions}人がCV（実測CVR: {conversions/visitors:.2%}）")
    print(f"推定CVR: E[θ|data] = α/(α+β) = {post_alpha}/({post_alpha}+{post_beta}) = {mean_rate:.2%}")
    print(f"         （事後分布の期待値 = このデータから推定される、本来のCVR）")
    print(f"95%信用区間: [{ci_lower:.2%}, {ci_upper:.2%}]")
    print(f"P(θ>{target_cvr:.1%}) = 1 - CDF_Beta({target_cvr}, α={post_alpha}, β={post_beta}) = 1 - {cdf_value:.4f} = {prob_above_target:.1%}")
    print(f"         （事後分布で目標を超える確率 = 本来のCVRが{target_cvr:.1%}以上である確率）")
    print(f"判定   : {prob_above_target:.1%} {'≥' if prob_above_target >= 0.95 else '<'} 95% → {verdict}")
    print(f"-" * 45)
    print()
    
    # 4. Generate Data for Plotting
    x_max = max(mean_rate * 3, target_cvr * 2, ci_upper * 1.5)
    x = np.linspace(0, x_max, 2000)
    y = beta.pdf(x, post_alpha, post_beta)
    y_max = max(y)
    
    # 5. Create Interactive Plotly Graph
    fig = go.Figure()

    # Define Theme Colors
    primary_blue = '#1f77b4'
    fill_blue = 'rgba(31, 119, 180, 0.3)'

    # A. The Main Curve
    fig.add_trace(go.Scatter(
        x=x, y=y,
        mode='lines',
        name='P(θ|data) Posterior',
        line=dict(color=primary_blue, width=3)
    ))

    # B. The Highlighted 95% Confidence Area
    mask = (x >= ci_lower) & (x <= ci_upper)
    fig.add_trace(go.Scatter(
        x=x[mask], y=y[mask],
        mode='lines',
        fill='tozeroy', 
        name='95% Credible Interval',
        line=dict(width=0),
        fillcolor=fill_blue
    ))

    # C. Vertical Lines (without built-in annotations to avoid overlap)
    fig.add_vline(x=target_cvr, line_width=3, line_dash="solid", line_color="#d62728")
    fig.add_vline(x=ci_lower, line_width=2, line_dash="dash", line_color=primary_blue)
    fig.add_vline(x=ci_upper, line_width=2, line_dash="dash", line_color=primary_blue)
    fig.add_vline(x=mean_rate, line_width=2, line_dash="dot", line_color="#005b96")

    # D. Annotations with explicit positions to avoid overlap
    # Target label
    fig.add_annotation(
        x=target_cvr, y=y_max * 0.95,
        text=f"Target: {target_cvr:.1%}",
        showarrow=True, arrowhead=2, arrowcolor="#d62728",
        ax=-60, ay=-30,
        font=dict(color="#d62728", size=12, weight='bold'),
        bgcolor="rgba(255,255,255,0.85)", bordercolor="#d62728", borderwidth=1, borderpad=3
    )

    # Mean label - E[θ|data]
    fig.add_annotation(
        x=mean_rate, y=y_max * 1.05,
        text=f"E[θ|data] = {mean_rate:.2%}",
        showarrow=False, yshift=10,
        font=dict(color="#005b96", size=12, weight='bold'),
        bgcolor="rgba(255,255,255,0.85)", bordercolor="#005b96", borderwidth=1, borderpad=3
    )

    # Lower bound label - 2.5th percentile of posterior θ
    fig.add_annotation(
        x=ci_lower, y=0,
        text=f"θ_2.5% = {ci_lower:.2%}",
        showarrow=True, arrowhead=2, arrowcolor=primary_blue,
        ax=-50, ay=35,
        font=dict(color=primary_blue, size=11),
        bgcolor="rgba(255,255,255,0.85)", bordercolor=primary_blue, borderwidth=1, borderpad=3
    )

    # Upper bound label - 97.5th percentile of posterior θ
    fig.add_annotation(
        x=ci_upper, y=0,
        text=f"θ_97.5% = {ci_upper:.2%}",
        showarrow=True, arrowhead=2, arrowcolor=primary_blue,
        ax=50, ay=35,
        font=dict(color=primary_blue, size=11),
        bgcolor="rgba(255,255,255,0.85)", bordercolor=primary_blue, borderwidth=1, borderpad=3
    )

    # 95% CI label - center of the shaded area
    fig.add_annotation(
        x=mean_rate, y=y_max * 0.4,
        text="95% Credible<br>Interval",
        showarrow=False,
        font=dict(color=primary_blue, size=12, weight='bold')
    )

    # 6. Layout Formatting
    fig.update_layout(
        title=dict(
            text=f"<b>Bayesian Fake Door Test</b><br>"
                 f"P(θ|data) ~ Beta(α={post_alpha}, β={post_beta})<br>"
                 f"P(θ > {target_cvr:.1%}) = {prob_above_target:.1%} → <b>{verdict}</b>",
            font=dict(size=20)
        ),
        xaxis_title="θ (Conversion Rate)",
        yaxis_title="P(θ|data) — Probability Density",
        xaxis=dict(tickformat=".1%", showgrid=True, gridcolor='#eee'),
        yaxis=dict(showgrid=True, gridcolor='#eee'),
        template="plotly_white",
        showlegend=True,
        legend=dict(y=0.99, x=0.99, bgcolor="rgba(255,255,255,0.8)"),
        font=dict(family="Arial, sans-serif"),
        plot_bgcolor='rgba(0,0,0,0)'
    )

    fig.show()

# ==========================================
# ENTER YOUR DATA HERE & RUN
# ==========================================
run_interactive_fake_door_nice_blue(visitors=2000, conversions=30, target_cvr=0.01)

--- フェイクドアテスト ---
手法   : ベイズ推定 ベータ二択分布 モデル）
         Beta = CVRのような確率の分布を表す型。
         Binomial = 「成功 or 失敗」の試行を繰り返すデータの型。
         → 訪問者が「CVするかしないか」のデータ(二択)から、
           真のCVRの確率分布(Beta)を推定するモデル。
         事前分布を無情報（一様分布）とし、観測データから
         真のCVRに対する信念を更新する。
KPI    : コンバージョン率（CVR）
目標   : 事業成立ライン 1.0%
---------------------------------------------
実績   : 2000人中 30人がCV（実測CVR: 1.50%）
推定CVR: E[θ|data] = α/(α+β) = 31/(31+1971) = 1.55%
         （事後分布の期待値 = このデータから推定される、本来のCVR）
95%信用区間: [1.05%, 2.13%]
P(θ>1.0%) = 1 - CDF_Beta(0.01, α=31, β=1971) = 1 - 0.0131 = 98.7%
         （事後分布で目標を超える確率 = 本来のCVRが1.0%以上である確率）
判定   : 98.7% ≥ 95% → 続行
---------------------------------------------

